# 02 — Data preparation, baselines and AutoML v0.1 (Week 2)

**Project:** NBA All-Star prediction — ML-2026, ESILV · **Modules 3 (modelling) and 4 (evaluation)** — owner: Basile Chabrel, reviewer: Augustin Nilsson

**Question answered here:** using only what is known *before* season *s* starts (season *s−1* statistics, age, experience, All-Star history), how well can we rank the players of season *s* by their chance of being selected as All-Stars?

**Rules of this notebook**
- Every random step uses `SEED` from `src/config.py`.
- Split by season, never at random: train 1981–2010, validation 2011–2017, test 2018–2026.
- Imputation, scaling and feature selection are fitted inside the sklearn `Pipeline`, i.e. on training rows only.
- All choices (model, hyperparameters, threshold) are made on **validation**. The test seasons are scored once, at the end.
- Accuracy is not reported as a performance metric (5–6 % positives).

In [1]:
import json
import sys
import warnings
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(PROJECT_ROOT))
warnings.filterwarnings("ignore", category=UserWarning)

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.calibration import calibration_curve
from sklearn.metrics import average_precision_score

from src.config import FIGURE_DPI, FIGURES_DIR, SEED, TABLES_DIR, TARGET_COL, ensure_dirs
from src.features.build_dataset import build_model_table
from src.models.split import temporal_split, describe_split, feature_columns
from src.models.automl_v01 import baseline_scores, run_automl, score_of
from src.evaluation.metrics import (best_f1_threshold, calibration_summary, hits_at_k,
                                    point_metrics, season_bootstrap_ci)

ensure_dirs()
np.random.seed(SEED)
pd.set_option("display.width", 160)

## 1. Build the modelling table

Implemented decisions (proposed defaults, see `src/features/build_dataset.py`; **to be confirmed with Augustin**):

| Decision | Choice | Why |
|---|---|---|
| D1 traded players | combined 2TM/3TM row | one row per label, no duplicated player across sets |
| D2 prediction time | **season s−1 statistics** (main); season s statistics only as a leakage demo | everything is known before the season: zero leakage |
| D3 replacements | every label row is positive | simplest; the `replaced` flag is inconsistent (Week 1) |
| D4 seasons | targets 1981–2026 (1999 has no label) | 3-point stats exist for every feature season |

In [2]:
table = build_model_table("prev")
leaky = build_model_table("full")
for name, t in [("prev (main)", table), ("full (leakage demo)", leaky)]:
    print(f"{name:20s} rows={len(t):,}  All-Stars={t[TARGET_COL].sum():,}  rate={t[TARGET_COL].mean():.2%}  "
          f"seasons={t.season.min()}-{t.season.max()} ({t.season.nunique()})")
assert not table.duplicated(["player_id", "season"]).any()
features = feature_columns(table)
print(len(features), "features:", features)

prev (main)          rows=19,823  All-Stars=1,158  rate=5.84%  seasons=1981-2026 (45)
full (leakage demo)  rows=19,823  All-Stars=1,158  rate=5.84%  seasons=1981-2026 (45)
45 features: ['age', 'experience', 'prev_g', 'prev_gs', 'prev_mp_per_game', 'prev_pts_per_game', 'prev_trb_per_game', 'prev_ast_per_game', 'prev_stl_per_game', 'prev_blk_per_game', 'prev_tov_per_game', 'prev_pf_per_game', 'prev_fga_per_game', 'prev_fta_per_game', 'prev_x3pa_per_game', 'prev_fg_percent', 'prev_x3p_percent', 'prev_ft_percent', 'prev_e_fg_percent', 'prev_per', 'prev_ts_percent', 'prev_usg_percent', 'prev_ast_percent', 'prev_trb_percent', 'prev_stl_percent', 'prev_blk_percent', 'prev_tov_percent', 'prev_ows', 'prev_dws', 'prev_ws', 'prev_ws_48', 'prev_obpm', 'prev_dbpm', 'prev_bpm', 'prev_vorp', 'prev_team_win_pct', 'prev_pts_per_game_season_rank', 'prev_ws_season_rank', 'prev_vorp_season_rank', 'prev_per_season_rank', 'prev_bpm_season_rank', 'prev_mp_per_game_season_rank', 'n_prev_all_star', 'all_star_p

## 2. Checks before modelling

Three things can silently break the model: missing values, rookies (no previous season), and the class imbalance.

In [3]:
missing = table[features].isna().mean().sort_values(ascending=False)
print("Share of missing values (top 8):")
print(missing.head(8).round(3).to_string())
rookies = table[table.has_prev_season == 0]
print(f"\nRows without a previous season: {len(rookies):,} ({len(rookies)/len(table):.1%}); "
      f"All-Stars among them: {rookies[TARGET_COL].sum()} -> {', '.join(rookies[rookies[TARGET_COL]==1].player + ' ' + rookies[rookies[TARGET_COL]==1].season.astype(str))}")
print(f"\nPositives per season: min={table.groupby('season')[TARGET_COL].sum().min()}, "
      f"max={table.groupby('season')[TARGET_COL].sum().max()}")

Share of missing values (top 8):
prev_x3p_percent     0.298
prev_gs              0.229
prev_ft_percent      0.221
prev_fg_percent      0.212
prev_e_fg_percent    0.212
prev_ts_percent      0.212
prev_tov_percent     0.212
prev_per             0.211

Rows without a previous season: 4,185 (21.1%); All-Stars among them: 20 -> Tim Duncan 1998, Kevin Durant 2021, Patrick Ewing 1986, Blake Griffin 2011, Grant Hill 1995, Grant Hill 2005, Michael Jordan 1985, Michael Jordan 2002, Yao Ming 2003, Yao Ming 2011, Dikembe Mutombo 1992, Hakeem Olajuwon 1985, Shaquille O'Neal 1993, David Robinson 1990, Ralph Sampson 1984, Isiah Thomas 1982, Kelly Tripucka 1982, Buck Williams 1982, Gus Williams 1982, Zion Williamson 2023

Positives per season: min=23, max=29


Missing values are mostly **structural**: ~21 % of rows have no previous season (rookies, players back from injury), and shooting percentages are empty when a player took no shot of that type. We impute with the training median **and** keep a missing-indicator column (`SimpleImputer(add_indicator=True)`), so the model still knows the value was absent.

20 All-Stars had no previous NBA season (e.g. Blake Griffin 2011, Yao Ming 2003): with D2 = "previous season" the model can hardly find them. This is a known limitation of the choice, not a bug.

In [4]:
# Quick look: which single features separate All-Stars best? (training seasons only)
train_only = table[table.season <= 2010]
auc_single = {f: average_precision_score(train_only[TARGET_COL], train_only[f].fillna(train_only[f].median()))
              for f in features}
corr = train_only[features + [TARGET_COL]].corr(numeric_only=True)[TARGET_COL].drop(TARGET_COL)
single = pd.DataFrame({"pr_auc_alone": auc_single, "pearson_r": corr}).sort_values("pr_auc_alone", ascending=False)
print(single.head(10).round(3).to_string())
high_corr = (table[features].corr().abs().where(lambda m: np.triu(np.ones(m.shape, bool), 1)).stack()
             .loc[lambda s: s > 0.9].sort_values(ascending=False))
print(f"\n{len(high_corr)} feature pairs with |r| > 0.9 (redundancy, handled by regularisation / selection), e.g.:")
print(high_corr.head(5).round(3).to_string())

                               pr_auc_alone  pearson_r
prev_vorp_season_rank                 0.624      0.369
prev_vorp                             0.613      0.584
prev_ws_season_rank                   0.609      0.375
prev_ws                               0.602      0.542
prev_pts_per_game_season_rank         0.590      0.384
prev_per_season_rank                  0.588      0.392
prev_pts_per_game                     0.587      0.513
prev_per                              0.571      0.433
prev_ows                              0.531      0.523
prev_bpm_season_rank                  0.528      0.370



11 feature pairs with |r| > 0.9 (redundancy, handled by regularisation / selection), e.g.:
prev_mp_per_game   prev_mp_per_game_season_rank    0.992
prev_pts_per_game  prev_fga_per_game               0.981
prev_ows           prev_ws                         0.946
prev_e_fg_percent  prev_ts_percent                 0.942
prev_obpm          prev_bpm                        0.939


## 3. Temporal split

In [5]:
parts = temporal_split(table)
split_table = describe_split(parts)
print(split_table.to_string(index=False))
split_table.to_csv(TABLES_DIR / "week2_split.csv", index=False)
train, validation, test = parts["train"], parts["validation"], parts["test"]

       set   seasons  n_seasons  rows  all_stars  positive_rate
     train 1981-2010         29 11482        738         0.0643
validation 2011-2017          7  3335        179         0.0537
      test 2018-2026          9  5006        241         0.0481


The positive rate falls from 6.4 % (train) to 4.8 % (test): more players per season, same number of All-Star places. A PR-AUC must always be read against this rate (the random baseline).

## 4. Baselines (validation seasons)

A model is only useful if it beats simple rules:
1. **random** score — PR-AUC ≈ positive rate;
2. **prev_points** — last season's top scorers;
3. **same_as_last_year** — last season's All-Stars first, then by points. This one is hard to beat: All-Star selection is very persistent.

In [6]:
rows = []
for name, s in baseline_scores(validation).items():
    frame = validation[["season", TARGET_COL]].assign(score=s)
    thr = best_f1_threshold(frame[TARGET_COL], frame.score)  # for baselines, illustrative only
    rows.append({"model": f"baseline: {name}", **point_metrics(frame, thr)})
baselines_val = pd.DataFrame(rows).set_index("model")
print(baselines_val.round(3).to_string())

                             pr_auc  roc_auc  precision  recall     f1  hits_at_k  positive_rate
model                                                                                           
baseline: random              0.050    0.478      0.054   1.000  0.103      0.039          0.054
baseline: prev_points         0.634    0.948      0.558   0.620  0.587      0.575          0.054
baseline: same_as_last_year   0.719    0.958      0.713   0.654  0.682      0.676          0.054


## 5. AutoML v0.1 — model selection on validation

Standard algorithms from the course, each with a small grid. Every candidate is fitted on train seasons and scored (PR-AUC) on validation seasons through a `PredefinedSplit`.
- Logistic regression and decision tree use `class_weight="balanced"` (imbalance).
- SVM uses its decision function as a score (no probabilities needed for PR-AUC).
- `SelectKBest` (mutual information) is part of the grid: feature selection is learned, not hand-picked.

In [7]:
automl = run_automl(train, validation)
searches = automl.attrs["searches"]
print(automl.to_string(index=False))
automl.assign(best_params=automl.best_params.astype(str)).to_csv(TABLES_DIR / "week2_automl_v01_validation.csv", index=False)

              model                                                                   best_params  validation_pr_auc
logistic_regression                                        {'model__C': 0.01, 'select__k': 'all'}           0.753260
                knn {'model__n_neighbors': 150, 'model__weights': 'distance', 'select__k': 'all'}           0.734405
      decision_tree    {'model__max_depth': 8, 'model__min_samples_leaf': 50, 'select__k': 'all'}           0.666212
            svm_rbf                      {'model__C': 0.1, 'model__gamma': 0.01, 'select__k': 15}           0.660842


In [8]:
best_name = automl.loc[0, "model"]
best_model = searches[best_name]
cols = feature_columns(train)
val_frame = validation[["season", "player", TARGET_COL]].assign(score=score_of(best_model, validation[cols]))
threshold = best_f1_threshold(val_frame[TARGET_COL], val_frame.score)
print(f"Selected v0.1 model: {best_name}  (validation PR-AUC {automl.loc[0, 'validation_pr_auc']:.3f})")
print(f"Decision threshold chosen on validation (max F1): {threshold:.3f}")
print(pd.Series(point_metrics(val_frame, threshold)).round(3).to_string())

Selected v0.1 model: logistic_regression  (validation PR-AUC 0.753)
Decision threshold chosen on validation (max F1): 0.820
pr_auc           0.753
roc_auc          0.975
precision        0.625
recall           0.782
f1               0.695
hits_at_k        0.676
positive_rate    0.054


### 5.1 Calibration of the selected model

`class_weight="balanced"` multiplies the weight of positives by ~15, so the raw probabilities are **inflated**: they rank players well but are not real probabilities. We check it on validation, and we fit a Platt (sigmoid) recalibration **on validation** so the final probabilities can be read as "chance of being an All-Star". The ranking (PR-AUC, hits@K) does not change.

In [9]:
from sklearn.calibration import CalibratedClassifierCV
from sklearn.frozen import FrozenEstimator

calibrated = CalibratedClassifierCV(FrozenEstimator(best_model), method="sigmoid").fit(validation[cols], validation[TARGET_COL])
raw_val = best_model.predict_proba(validation[cols])[:, 1]
fig, ax = plt.subplots(figsize=(5.5, 5))
for label, proba in [("raw (balanced weights)", raw_val)]:
    frac, mean = calibration_curve(validation[TARGET_COL], proba, n_bins=10, strategy="quantile")
    ax.plot(mean, frac, "o-", label=label)
ax.plot([0, 1], [0, 1], "--", color="grey", label="perfect calibration")
ax.set_xlabel("Mean predicted probability"); ax.set_ylabel("Observed All-Star rate")
ax.set_title(f"Reliability on validation seasons — {best_name}"); ax.legend()
fig.tight_layout(); fig.savefig(FIGURES_DIR / "fig5_calibration_validation.png", dpi=FIGURE_DPI); plt.show()
print("Raw on validation:", {k: round(v, 4) for k, v in calibration_summary(validation[TARGET_COL], raw_val).items()})
cal_test = calibrated.predict_proba(test[cols])[:, 1]
raw_test = best_model.predict_proba(test[cols])[:, 1]
print("Test, raw       :", {k: round(v, 4) for k, v in calibration_summary(test[TARGET_COL], raw_test).items()})
print("Test, calibrated:", {k: round(v, 4) for k, v in calibration_summary(test[TARGET_COL], cal_test).items()})

Raw on validation: {'brier': 0.0599, 'mean_predicted': 0.1688, 'observed_rate': 0.0537}


Test, raw       : {'brier': 0.0567, 'mean_predicted': 0.1596, 'observed_rate': 0.0481}
Test, calibrated: {'brier': 0.0257, 'mean_predicted': 0.0515, 'observed_rate': 0.0481}


## 6. Final evaluation of v0.1 on the test seasons (2018–2026), touched once

95 % confidence intervals: season-block bootstrap (1,000 resamples of whole seasons).

In [10]:
results = {}
test_frame = test[["season", "player", TARGET_COL]].assign(score=score_of(best_model, test[cols]))
results[f"v0.1 {best_name}"] = season_bootstrap_ci(test_frame, threshold)
for name, s in baseline_scores(test).items():
    if name == "random":
        continue
    f = test[["season", TARGET_COL]].assign(score=s)
    thr = best_f1_threshold(validation[TARGET_COL], baseline_scores(validation)[name])  # threshold from validation
    results[f"baseline: {name}"] = season_bootstrap_ci(f, thr)
summary = pd.concat({k: v.T.loc["value"] for k, v in results.items()}, axis=1).T
ci = pd.concat({k: v.apply(lambda r: f"[{r.ci_low:.3f}, {r.ci_high:.3f}]", axis=1) for k, v in results.items()}, axis=1).T
print("Point estimates on test:"); print(summary.round(3).to_string())
print("\n95% CI:"); print(ci[["pr_auc", "f1", "hits_at_k"]].to_string())
summary.round(4).to_csv(TABLES_DIR / "week2_test_results.csv")
ci.to_csv(TABLES_DIR / "week2_test_ci.csv")

Point estimates on test:
                             pr_auc  roc_auc  precision  recall     f1  hits_at_k  positive_rate
v0.1 logistic_regression      0.678    0.975      0.540   0.734  0.622      0.614          0.048
baseline: prev_points         0.628    0.961      0.425   0.817  0.560      0.602          0.048
baseline: same_as_last_year   0.631    0.963      0.610   0.564  0.586      0.614          0.048

95% CI:
                                     pr_auc              f1       hits_at_k
v0.1 logistic_regression     [0.636, 0.720]  [0.591, 0.658]  [0.577, 0.650]
baseline: prev_points        [0.577, 0.693]  [0.531, 0.593]  [0.555, 0.650]
baseline: same_as_last_year  [0.584, 0.688]  [0.531, 0.637]  [0.554, 0.661]


In [11]:
hits = hits_at_k(test_frame)
print(hits.to_string(index=False))
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.bar(hits.season.astype(str), hits.hit_rate, color="#2a6f97")
ax.axhline(hits.hits.sum() / hits.k.sum(), color="#e07a1f", ls="--", label="overall")
ax.set_ylim(0, 1); ax.set_ylabel("Share of All-Stars found in top-K"); ax.set_xlabel("Test season")
ax.set_title(f"Hits@K per test season — v0.1 {best_name}"); ax.legend()
fig.tight_layout(); fig.savefig(FIGURES_DIR / "fig6_hits_at_k_test_v01.png", dpi=FIGURE_DPI); plt.show()

 season  k  hits  hit_rate
   2018 28    19  0.678571
   2019 27    17  0.629630
   2020 25    13  0.520000
   2021 27    19  0.703704
   2022 27    17  0.629630
   2023 27    15  0.555556
   2024 26    16  0.615385
   2025 26    16  0.615385
   2026 28    16  0.571429


## 7. Leakage demonstration: same pipeline with season-s statistics

Same algorithm and hyperparameters, but the features are the statistics of the **same** season, which include games played after the All-Star selection (January). This is the "retrospective" option D2 = a. The gap between the two numbers is the optimism that leakage would add — one of the AI-audit cases (an LLM proposed this setup first; see `ai_governance/AI_AUDIT.md`).

In [12]:
from sklearn.base import clone
leaky_parts = temporal_split(leaky)
lcols = feature_columns(leaky)
leaky_model = clone(best_model).fit(leaky_parts["train"][lcols], leaky_parts["train"][TARGET_COL])
leaky_test = leaky_parts["test"][["season", TARGET_COL]].assign(score=score_of(leaky_model, leaky_parts["test"][lcols]))
lthr = best_f1_threshold(leaky_parts["validation"][TARGET_COL], score_of(leaky_model, leaky_parts["validation"][lcols]))
leak_cmp = pd.DataFrame({
    "no leakage (season s-1 stats)": point_metrics(test_frame, threshold),
    "leaky (season s stats)": point_metrics(leaky_test, lthr),
}).T[["pr_auc", "roc_auc", "f1", "hits_at_k"]]
print(leak_cmp.round(3).to_string())
leak_cmp.round(4).to_csv(TABLES_DIR / "week2_leakage_comparison.csv")

                               pr_auc  roc_auc     f1  hits_at_k
no leakage (season s-1 stats)   0.678    0.975  0.622      0.614
leaky (season s stats)          0.852    0.991  0.775      0.768


## 8. Risks, failure modes and plan for Week 3

| Risk / failure mode | Evidence | Plan |
|---|---|---|
| Rookies and returning players are invisible | 20 All-Stars without a previous season | report them separately; an "All-Star break" version (D2 = b, needs `nba_api`) would fix it |
| Popularity and narrative, not stats | voters = 50 % fans | All-Star history feature; error analysis by player in Week 3 |
| Temporal drift | positive rate 6.4 % → 4.8 %; pace and 3-point era | within-season rank features; drift experiment in Week 3 |
| Threshold instability | threshold chosen on 7 validation seasons | report threshold-free metrics (PR-AUC, hits@K) first |
| Leakage | season-s stats inflate results (section 7) | main model uses season s−1 only |

Week 3 (v0.2): random forest, gradient boosting, an MLP (outside the course, justified with Grinsztajn et al., NeurIPS 2022), rank-average ensemble, permutation importance, drift experiment, 2026 case study.